# Training the PINN model

## 0. Prerrequesites

In [1]:
import sys
sys.path.append("/scratchsan/observatorio/juagudeloo/Tesis_maestria_OAN/")

from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import astropy.units as u
import matplotlib.pyplot as plt

from utils.muram_data import MhdData, StokesData
from utils.normalizer import MhdNormalizer, StokesNormalizer
from models.pinn_mscnn_model import PhysicsInformedMSCNN
from utils.physics_utils import ApproxInversions

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Initialization

### 1.1 Charging data

In [2]:
# Data paths (adjust to your system)
data_path = Path("/scratchsan/observatorio/juagudeloo/data/")
step = 112  # Choose a simulation step

# Load MHD data and remap to optical depth
mhd = MhdData(
    data_path=data_path / "muram-simulation",
    nx=480, ny=480, nz=256
)
mhd.load_step(step=step, z_max=250)
mhd.load_opacity_table(kappa_path=data_path / "csv/kappa.0.dat")
mhd.compute_optical_depth(dz=10*u.km)

# Create a uniform log(tau) grid for remapping
new_logtau = np.arange(-2.0, 0.1, 0.1)
mhd.remap_to_optical_depth(new_logtau, quantities=["T", "Vz", "Bz"])

print(f"MHD data loaded: {mhd.od_data['Bz'].shape}")
print(f"Log(tau) values: {new_logtau}")

# Load Stokes data
stokes = StokesData(
    data_dir=data_path / "muram-simulation/",
    step=step,
    wavelength_range=(6300.5, 6303.5),
    wavelength_step=0.01
)
stokes.load_stokes()
stokes.continuum_normalization(cont_indices=[0, 1, 2, 3])
stokes.load_hinode_lsf(data_path / "hinode-MODEST/PSFs/hinode_sp.spline.psf")
stokes.apply_spectral_convolution()
stokes.resample_to_hinode()

print(f"Stokes data loaded: {stokes.data['I'].shape}")
print(f"Stokes wavelength range: {stokes.wl.min():.2f} - {stokes.wl.max():.2f} Å")

Loading step 112000


  Trimmed z axis to first 250 layers (0..249)
  Reference layer (T ~ 5780 K) at z-index 186
  Done.
Opacity interpolator loaded from /scratchsan/observatorio/juagudeloo/data/csv/kappa.0.dat
  T range: [3.320, 5.300] (log10 K)
  P range: [-2.000, 8.000] (log10 dyne/cm²)
Computing optical depth...
  Optical depth computed.
Remapping to optical depth coordinates (21 levels)...
  Processing T
  Processing Vz
  Processing Bz
  Remapping complete.
MHD data loaded: (480, 480, 21)
Log(tau) values: [-2.00000000e+00 -1.90000000e+00 -1.80000000e+00 -1.70000000e+00
 -1.60000000e+00 -1.50000000e+00 -1.40000000e+00 -1.30000000e+00
 -1.20000000e+00 -1.10000000e+00 -1.00000000e+00 -9.00000000e-01
 -8.00000000e-01 -7.00000000e-01 -6.00000000e-01 -5.00000000e-01
 -4.00000000e-01 -3.00000000e-01 -2.00000000e-01 -1.00000000e-01
  1.77635684e-15]
Loading Stokes data from /scratchsan/observatorio/juagudeloo/data/muram-simulation/stokes_112000.npy
  I shape: (480, 480, 300)
  Q shape: (480, 480, 300)
  U sha

### 1.2 Model instantiation

In [3]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Instantiate the Physics-Informed MSCNN model
model = PhysicsInformedMSCNN(
    scales=[1, 2, 3],                           # Multi-scale coarse-graining
    in_channels=2,                              # I and V Stokes parameters
    c1_filters=16,                              # Filters in first conv block
    c2_filters=32,                              # Filters in second conv block
    kernel_size=5,                              # Convolution kernel size
    pool_size=2,                                # MaxPool kernel size
    n_linear_layers=4,                          # Dense layers for final mapping
    output_features=3*21,                       # 3 parameters × 21 heights = 63 outputs
    input_length=112,                           # Spectral dimension of Stokes profiles
    
    # Physics-informed parameters
    central_wavelength=6301.5*u.Angstrom,      # Fe I 6301.5 Å
    lande_factor=1.67,                          # Landé g-factor
    wl_range=(15, 60),                          # Wavelength indices for WFA/Doppler
    lambda_reg=0.1,                             # Regularization weight
    use_physics='both',                         # Use both WFA and Doppler (can be None, 'wfa', 'doppler', or 'both')

    # Dropout for uncertainty
    dropout_rate=0.2  # 20% dropout
).to(device)

print(f"Model instantiated on {device}")
print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Regularization weight (lambda_reg): {model.lambda_reg}")
print(f"Physics mode: {model.use_physics}")

Using device: cuda
Model instantiated on cuda
Total trainable parameters: 7,318,223
Regularization weight (lambda_reg): 0.1
Physics mode: both


## 2. Data normalization

### 2.1 MHD Normalization

The normalization scheme applies **parameter-specific transformations** independently at **each optical depth level** τ. This ensures equal contribution from all atmospheric heights during training. 



#### 2.1.2 Mathematical Formulation

##### *Temperature Normalization (Standard Scaling)*

For each optical depth level $\tau_k$ where $k \in \{0, 1, ..., 20\}$:

$$
T_{\text{norm}}(x, y, \tau_k) = \frac{T(x, y, \tau_k) - \mu_T(\tau_k)}{\sigma_T(\tau_k) + \epsilon}
$$

**Where:**
- $T(x, y, \tau_k)$: Temperature at spatial position $(x, y)$ and optical depth $\tau_k$ [Kelvin]
- $\mu_T(\tau_k)$: Mean temperature computed across all simulation steps at depth $\tau_k$
- $\sigma_T(\tau_k)$: Standard deviation at depth $\tau_k$
- $\epsilon = 10^{-8}$: Small constant to prevent division by zero

**Physical Interpretation:**
- Temperature stratifies strongly: $T(\tau=-2) \approx 4000$ K (high photosphere) to $T(\tau=0) \approx 7000$ K (deep photosphere)
- Per-$\tau$ normalization ensures the model learns temperature variations at all heights equally

---
#### *Line-of-Sight Velocity Normalization (Centered Scaling)*

For each optical depth level $\tau_k$:

$$
V_{z,\text{norm}}(x, y, \tau_k) = \frac{V_z(x, y, \tau_k)}{\sigma_{V_z}(\tau_k) + \epsilon}
$$

**Where:**
- $V_z(x, y, \tau_k)$: Line-of-sight velocity [km/s]
- $\sigma_{V_z}(\tau_k)$: Standard deviation at depth $\tau_k$
- **No mean subtraction** because velocity is naturally centered around zero (upflows: $V_z < 0$, downflows: $V_z > 0$)

**Physical Interpretation:**
- Convective velocities: $|V_z| \lesssim 5$ km/s
- Preserves the sign information (critical for flow direction)
- Velocity amplitude varies with height due to density stratification

---
#### *Magnetic Field Normalization (Signum-Log + Standard Scaling)*

This is a **two-step transformation** to handle the extreme dynamic range of magnetic fields.

#### Step 3a: Signum-Logarithmic Transform

$$
B_{z,\text{log}}(x, y, \tau_k) = \text{sgn}(B_z) \cdot \log_{10}\left(|B_z(x, y, \tau_k)| + 1\right)
$$

**Where:**
- $\text{sgn}(B_z) = \begin{cases} +1 & \text{if } B_z > 0 \\ -1 & \text{if } B_z < 0 \\ 0 & \text{if } B_z = 0 \end{cases}$
- The $+1$ offset handles zero fields: $\log_{10}(0 + 1) = 0$

**Why Logarithm?**
Magnetic fields span **orders of magnitude**:
- Quiet Sun: $|B_z| \sim 1-10$ G (intergranular network)
- Active regions: $|B_z| \sim 100-1500$ G (magnetic elements)

The log transform compresses this range while preserving polarity:

| Physical $B_z$ [G] | After Signum-Log |
|-------------------|------------------|
| 0                 | 0                |
| +10               | +1.04            |
| +100              | +2.00            |
| +1000             | +3.00            |
| -100              | -2.00            |

##### Step 3b: Standard Scaling (Per-τ)

$$
B_{z,\text{norm}}(x, y, \tau_k) = \frac{B_{z,\text{log}}(x, y, \tau_k) - \mu_{B_z}(\tau_k)}{\sigma_{B_z}(\tau_k) + \epsilon}
$$

**Where:**
- $\mu_{B_z}(\tau_k)$, $\sigma_{B_z}(\tau_k)$: Statistics computed on **log-transformed** values
- Magnetic field strength decreases with height: $|B_z(\tau=-2)| < |B_z(\tau=0)|$

---

#### Inverse Transformation (Denormalization)

During inference, model predictions must be converted back to physical units:

##### *Temperature (Inverse Standard Scaling)*

$$
T(x, y, \tau_k) = T_{\text{norm}}(x, y, \tau_k) \cdot \sigma_T(\tau_k) + \mu_T(\tau_k)
$$

##### *Velocity (Inverse Centered Scaling)*

$$
V_z(x, y, \tau_k) = V_{z,\text{norm}}(x, y, \tau_k) \cdot \sigma_{V_z}(\tau_k)
$$

##### *Magnetic Field (Inverse Signum-Log)*

$$
B_z(x, y, \tau_k) = \text{sgn}(B_{z,\text{norm}}') \cdot \left(10^{|B_{z,\text{norm}}'|} - 1\right)
$$

**Where:**

$$
B_{z,\text{norm}}' = B_{z,\text{norm}}(x, y, \tau_k) \cdot \sigma_{B_z}(\tau_k) + \mu_{B_z}(\tau_k)
$$

---

### Statistics Computation (Welford's Online Algorithm)

To handle memory constraints when processing 164 simulation steps, statistics are computed **incrementally**:

#### Initialization
For each parameter $p \in \{T, V_z, B_z\}$ and each $\tau_k$:

$$
n^{(p)}_k = 0, \quad \mu^{(p)}_k = 0, \quad M_2^{(p)}_k = 0
$$

#### Update Rule
For each new data point $x$:

$$
\begin{align}
n_k &\leftarrow n_k + 1 \\
\delta &= x - \mu_k \\
\mu_k &\leftarrow \mu_k + \frac{\delta}{n_k} \\
\delta' &= x - \mu_k \quad \text{(updated mean)} \\
M_{2,k} &\leftarrow M_{2,k} + \delta \cdot \delta'
\end{align}
$$

#### Finalization

$$
\sigma_k = \sqrt{\frac{M_{2,k}}{n_k}}
$$

**Properties:**
- Mathematically equivalent to batch computation
- Memory-efficient: $\mathcal{O}(1)$ storage per $(\tau_k, \text{parameter})$
- Numerically stable (avoids catastrophic cancellation)

---

### Model Input/Output Format

### Input (Stokes Profiles)
$$
\mathbf{x} \in \mathbb{R}^{2 \times 112}
$$
- 2 channels: Stokes $I$ (intensity), Stokes $V$ (circular polarization)
- 112 wavelength samples: $\lambda \in [6300.5, 6303.5]$ Å with Hinode sampling

#### Output (Normalized Atmospheric Parameters)
$$
\mathbf{y} \in \mathbb{R}^{63}
$$

Concatenated as:
$$
\mathbf{y} = \begin{bmatrix}
T_{\text{norm}}(\tau_0), \ldots, T_{\text{norm}}(\tau_{20}) \\
V_{z,\text{norm}}(\tau_0), \ldots, V_{z,\text{norm}}(\tau_{20}) \\
B_{z,\text{norm}}(\tau_0), \ldots, B_{z,\text{norm}}(\tau_{20})
\end{bmatrix}
$$

**Ordering:** $[T(\tau_0), \ldots, T(\tau_{20}), V_z(\tau_0), \ldots, V_z(\tau_{20}), B_z(\tau_0), \ldots, B_z(\tau_{20})]$

---

### Expected Value Ranges After Normalization

For well-normalized data (across all $\tau_k$):

| Parameter | Expected Range | Mean | Std Dev |
|-----------|---------------|------|---------|
| $T_{\text{norm}}$ | $[-3, +3]$ | $\approx 0$ | $\approx 1$ |
| $V_{z,\text{norm}}$ | $[-4, +4]$ | $\approx 0$ | $\approx 1$ |
| $B_{z,\text{norm}}$ | $[-3, +3]$ | $\approx 0$ | $\approx 1$ |

Values outside $[-5, +5]$ indicate potential outliers or out-of-distribution data.

---

### Advantages of Per-τ Normalization

1. **Equal Loss Contribution**: All atmospheric heights contribute equally to gradients
2. **Physical Stratification**: Respects that $T$, $V_z$, $B_z$ vary systematically with height
3. **Numerical Stability**: Prevents deep photosphere from dominating the loss function
4. **Generalization**: Model learns height-dependent physics rather than absolute scales

In [4]:
# Load intermediate state
mhd_normalizer = MhdNormalizer()
mhd_normalizer.load(data_path / "normalization_stats/mhd_normalization.json")

normalized_mhd = mhd_normalizer.transform(mhd.od_data)

Per-τ normalization statistics loaded from /scratchsan/observatorio/juagudeloo/data/normalization_stats/mhd_normalization.json
Optical depth levels: 21


In [5]:
for key in normalized_mhd:
    print(f"{key} shape: {normalized_mhd[key].shape}, min: {normalized_mhd[key].min():.3f}, max: {normalized_mhd[key].max():.3f}")

T shape: (480, 480, 21), min: -8.218, max: 5.088
Vz shape: (480, 480, 21), min: -3.758, max: 4.840
Bz shape: (480, 480, 21), min: -3.001, max: 3.047


### 2.2 Stokes Normalization

The normalization of Stokes parameters follows a **global standardization approach** computed across all simulation steps. Unlike atmospheric parameters that require per-optical-depth normalization due to stratification, Stokes I and V profiles must preserve their **temporal evolution** and **relative magnitudes** across the simulation.

---

#### Physical Motivation

##### Why Global Normalization?

The MURaM simulation evolves from an initial state with **weak magnetic fields** (early steps: 60–100) to a **magnetoconvective equilibrium** with strong flux concentrations (later steps: 150–223). The Stokes V signal amplitude directly correlates with this evolution:

- **Step 60**: $|V/I_c| \sim 10^{-4}$ (weak internetwork fields)
- **Step 200**: $|V/I_c| \sim 10^{-2}$ (kilogauss flux tubes)

**Per-step normalization would fail** because:
1. Each step would be normalized to the same range (e.g., $[-1, +1]$)
2. The model would lose the ability to distinguish weak vs. strong field regimes
3. Training would be biased toward later steps with larger gradients

**Global normalization preserves**:
- The **physical relationship** between V amplitude and field strength
- The **evolutionary context** of magnetoconvection
- **Balanced gradient contributions** from all simulation phases

---

#### Mathematical Formulation

##### Stokes I Normalization (Z-Score Standardization)

For Stokes I intensity profiles:

$$
I_{\text{norm}}(x, y, \lambda) = \frac{I(x, y, \lambda) - \mu_I}{\sigma_I + \epsilon}
$$

**Where:**
- $I(x, y, \lambda)$: Continuum-normalized intensity at spatial position $(x, y)$ and wavelength $\lambda$ [dimensionless]
- $\mu_I$: Global mean intensity computed across **all simulation steps and wavelengths**
- $\sigma_I$: Global standard deviation across all steps
- $\epsilon = 10^{-8}$: Numerical stability constant

**Physical Interpretation:**
- After continuum normalization, $I \approx 1$ in the continuum and $I \approx 0.2$–$0.8$ in line cores
- $\mu_I \approx 0.82$: Reflects the mean absorption across Fe I 6301.5 Å and 6302.5 Å lines
- $\sigma_I \approx 0.15$: Captures granulation contrast and line depth variations

---

##### Stokes V Normalization (Centered Scaling)

For circular polarization profiles:

$$
V_{\text{norm}}(x, y, \lambda) = \frac{V(x, y, \lambda)}{\sigma_V + \epsilon}
$$

**Where:**
- $V(x, y, \lambda)$: Continuum-normalized circular polarization [dimensionless]
- $\sigma_V$: Global standard deviation across all steps
- **No mean subtraction**: $V$ is naturally centered around zero (positive/negative lobes encode field polarity)

**Physical Interpretation:**
- Stokes V is an **antisymmetric profile** with $\langle V \rangle \approx 0$ (equal positive/negative field regions)
- $\sigma_V \approx 0.004$–$0.006$: Typical V amplitude for mixed polarity regions with $\langle |B_{\text{LOS}}| \rangle \sim 100$–$300$ G
- Preserves the **V/I ratio**, which is proportional to field strength via the Weak Field Approximation (WFA)

---

#### Statistics Computation (Welford's Online Algorithm)

To handle the massive dataset (164 steps × 480×480 pixels × 112 wavelengths ≈ 4.3 billion samples), statistics are computed **incrementally** without loading all data into memory.

##### Initialization

For each Stokes parameter $S \in \{I, V\}$:

$$
n^{(S)} = 0, \quad \mu^{(S)} = 0, \quad M_2^{(S)} = 0
$$

##### Update Rule

For each new sample $x$ (e.g., a single pixel intensity at one wavelength from one step):

$$
\begin{align}
n &\leftarrow n + 1 \\
\delta &= x - \mu \\
\mu &\leftarrow \mu + \frac{\delta}{n} \\
\delta' &= x - \mu \quad \text{(updated mean)} \\
M_2 &\leftarrow M_2 + \delta \cdot \delta'
\end{align}
$$

##### Finalization

After processing all 164 simulation steps:

$$
\sigma = \sqrt{\frac{M_2}{n}}
$$

**Properties:**
- **Memory-efficient**: Only stores 3 scalars per Stokes parameter ($n$, $\mu$, $M_2$)
- **Numerically stable**: Avoids catastrophic cancellation in variance computation
- **Exact**: Mathematically equivalent to batch computation of $\sigma = \sqrt{\mathbb{E}[(X - \mu)^2]}$

---

#### Implementation Details

##### Continuum Normalization (Pre-processing)

Before computing global statistics, each Stokes cube is continuum-normalized:

$$
I_{\text{cont-norm}}(x, y, \lambda) = \frac{I(x, y, \lambda)}{I_c(x, y)}
$$

$$
V_{\text{cont-norm}}(x, y, \lambda) = \frac{V(x, y, \lambda)}{I_c(x, y)}
$$

**Where:**
- $I_c(x, y) = \text{mean}(I(x, y, \lambda_{\text{cont}}))$: Continuum intensity estimated from the first 4 wavelength points
- This removes spatial intensity variations due to limb darkening and granulation

##### Spectral Degradation

To match **Hinode/SOT-SP** observational characteristics, the synthetic Stokes profiles undergo:

1. **Line Spread Function (LSF) Convolution**:
   $$
   I_{\text{conv}}(\lambda) = (I * \text{LSF})(\lambda)
   $$
   Applied with the Hinode SP spline PSF (FWHM ≈ 30 mÅ)

2. **Spectral Resampling**:
   From 300 points (10 mÅ sampling) → 112 points (21.5 mÅ sampling)
   $$
   \lambda_{\text{Hinode}} = \lambda_0 + (i - i_0) \cdot \Delta\lambda, \quad i = 1, \ldots, 112
   $$
   Where $\lambda_0 = 6302.0$ Å, $\Delta\lambda = 0.0215$ Å, $i_0 = 57$

These transformations are applied **before** computing normalization statistics to ensure the model trains on realistic data.

---

#### Model Input Format

##### Input (Normalized Stokes Profiles)

$$
\mathbf{x} \in \mathbb{R}^{2 \times 112}
$$

Constructed as:

$$
\mathbf{x} = \begin{bmatrix}
I_{\text{norm}}(\lambda_1), & I_{\text{norm}}(\lambda_2), & \ldots, & I_{\text{norm}}(\lambda_{112}) \\
V_{\text{norm}}(\lambda_1), & V_{\text{norm}}(\lambda_2), & \ldots, & V_{\text{norm}}(\lambda_{112})
\end{bmatrix}
$$

- **Channel 0**: Normalized intensity profile (absorption line shape)
- **Channel 1**: Normalized circular polarization (magnetic field signature)
- **Wavelength range**: $\lambda \in [6300.79, 6303.19]$ Å (Hinode/SP window)

---

#### Expected Value Ranges After Normalization

For well-normalized data across all simulation steps:

| Parameter | Expected Range | Mean | Std Dev | Physical Range (Pre-Norm) |
|-----------|---------------|------|---------|---------------------------|
| $I_{\text{norm}}$ | $[-3, +3]$ | $\approx 0$ | $\approx 1$ | $I \in [0.2, 1.0]$ (continuum-normalized) |
| $V_{\text{norm}}$ | $[-5, +5]$ | $\approx 0$ | $\approx 1$ | $V/I_c \in [-0.02, +0.02]$ |

**Interpretation:**
- **$I_{\text{norm}} \in [-3, +3]$**: Covers from deep line cores ($I \approx 0.2$) to continuum ($I \approx 1.0$)
- **$V_{\text{norm}} \in [-5, +5]$**: Captures weak fields ($|V| \sim 10^{-4}$) to kilogauss elements ($|V| \sim 10^{-2}$)
- Values outside $[-6, +6]$ indicate potential outliers or unrealistic Stokes profiles

---

#### Relationship to Physics-Informed Loss

The normalized Stokes V profiles directly constrain the model's predictions via the **Weak Field Approximation (WFA)**:

$$
V(\lambda) \approx -\frac{e}{4\pi m_e c} \lambda_0^2 g B_{\text{LOS}} \frac{dI}{d\lambda}
$$

After normalization, the WFA-derived magnetic field becomes:

$$
B_{\text{LOS}}^{\text{WFA}} = -\frac{V_{\text{norm}} \cdot \sigma_V}{\mathcal{C} \lambda_0^2 g \, dI_{\text{norm}}/d\lambda}
$$

Where $\mathcal{C} = e / (4\pi m_e c) = 4.67 \times 10^{-13}$ G$^{-1}$Å$^{-1}$ is the Zeeman splitting constant.

**Key point**: By preserving the **V/I ratio** through global normalization, the physics-informed loss correctly penalizes predictions inconsistent with Zeeman splitting.

---

#### Inverse Transformation (Denormalization)

During inference, normalized Stokes predictions must be converted back to physical units:

##### Stokes I (Inverse Z-Score)

$$
I(x, y, \lambda) = I_{\text{norm}}(x, y, \lambda) \cdot \sigma_I + \mu_I
$$

##### Stokes V (Inverse Centered Scaling)

$$
V(x, y, \lambda) = V_{\text{norm}}(x, y, \lambda) \cdot \sigma_V
$$

**Note**: If the model is used for **Stokes synthesis** (not implemented here), the output would be in continuum-normalized units ($I/I_c$, $V/I_c$). The absolute continuum level $I_c$ can be estimated from the mean of the first few wavelength points in $I(\lambda)$.

---

#### Advantages of Global Stokes Normalization

1. **Temporal Consistency**: All simulation phases (weak → strong fields) contribute equally to defining "typical" Stokes signals
2. **Physical Interpretability**: $\sigma_V$ represents the RMS polarization across the full magnetoconvective cycle
3. **Gradient Balance**: Prevents early steps (weak V) from being ignored due to small gradients
4. **WFA Compatibility**: Preserves the $V \propto B_{\text{LOS}} \cdot dI/d\lambda$ relationship needed for physics regularization
5. **Generalization**: Model learns to map Stokes amplitudes → field strengths across the full dynamic range ($B_{\text{LOS}} \in [0, 1500]$ G)

---

#### Verification Procedure

After computing normalization statistics, verify correctness with:

```python
# Load statistics
with open("stokes_normalization_stats.json", 'r') as f:
    stats = json.load(f)

# Apply normalization to a test step
stokes_norm = {
    'I': (stokes['I'] - stats['I']['mean']) / stats['I']['std'],
    'V': stokes['V'] / stats['V']['std']
}

# Check: Should be ~N(0,1)
print(f"I_norm: mean={stokes_norm['I'].mean():.4f}, std={stokes_norm['I'].std():.4f}")
print(f"V_norm: mean={stokes_norm['V'].mean():.4f}, std={stokes_norm['V'].std():.4f}")

# Check: Original V/I ratio preserved
ratio_original = stokes['V'].std() / stokes['I'].std()
ratio_normalized = stokes_norm['V'].std() / stokes_norm['I'].std()
print(f"V/I ratio: original={ratio_original:.4e}, normalized={ratio_normalized:.4e}")
# Should be approximately equal
```

**Expected output:**
```
I_norm: mean=0.0000, std=1.0000
V_norm: mean=0.0000, std=1.0000
V/I ratio: original=0.0029, normalized=0.0029
```

This confirms:
- ✓ Both parameters are properly standardized
- ✓ The physical V/I ratio is preserved
- ✓ Model will receive balanced inputs for learning

In [6]:
# Load intermediate state
stokes_normalizer = StokesNormalizer()
stokes_normalizer.load(data_path / "normalization_stats/stokes_normalization.json")

normalized_stokes = stokes_normalizer.transform(stokes.data)


Stokes normalization statistics loaded from /scratchsan/observatorio/juagudeloo/data/normalization_stats/stokes_normalization.json


In [7]:
for key in normalized_stokes:
    print(f"{key} shape: {normalized_stokes[key].shape}, quantile 1: {np.quantile(normalized_stokes[key], 0.01):.3f}, quantile 99: {np.quantile(normalized_stokes[key], 0.99):.3f} max: {np.max(normalized_stokes[key]):.3f} min: {np.min(normalized_stokes[key]):.3f}")

I shape: (480, 480, 112), quantile 1: -2.904, quantile 99: 1.792 max: 2.230 min: -3.393
V shape: (480, 480, 112), quantile 1: -1.550, quantile 99: 1.576 max: 50.944 min: -53.856


## 3. Tensor creation

In [8]:
# Step 1: Select a patch
y0, x0 = 100, 200
H, W = 4, 4
n_heights = 21

In [9]:
stokes_patch = {
    'I': normalized_stokes['I'][y0:y0+H, x0:x0+W, :],
    'V': normalized_stokes['V'][y0:y0+H, x0:x0+W, :],
}

# Step 3: Run model on the patch (batched)
print("\nRunning model inference on patch...")
I_p = torch.from_numpy(stokes_patch['I']).float().to(device)
V_p = torch.from_numpy(stokes_patch['V']).float().to(device)
patch_input = torch.stack([I_p, V_p], dim=2).view(H * W, 2, -1)
print(f"Input patched of shape: {patch_input.shape}")


Running model inference on patch...
Input patched of shape: torch.Size([16, 2, 112])


In [10]:
# Step 7: Build MHD patch cubes for height selection
mhd_patch_od_data = {
    'T': normalized_mhd['T'][y0:y0+H, x0:x0+W, :],
    'Bz': normalized_mhd['Bz'][y0:y0+H, x0:x0+W, :],
    'Vz': normalized_mhd['Vz'][y0:y0+H, x0:x0+W, :],
}

T_p = torch.from_numpy(mhd_patch_od_data['T'])
Bz_p = torch.from_numpy(mhd_patch_od_data['Bz'])
Vz_p = torch.from_numpy(mhd_patch_od_data['Vz'])

patched_targets = torch.stack([T_p, Bz_p, Vz_p], dim=3).reshape(H * W, n_heights*3)
print(f"Patched targets shape: {patched_targets.shape}")

Patched targets shape: torch.Size([16, 63])


In [11]:
print("Pre-computing physical approximations (WFA B_LOS and Doppler V_LOS) on patch...")

approx_invs = ApproxInversions(
    stokes=stokes.data,
    wavelength=stokes.wl,
    central_wavelength=model.central_wavelength,
    lande_factor=model.lande_factor,
)
blos_approx = approx_invs.compute_blos_wfa(wl_range=model.wl_range).value
vlos_approx = approx_invs.compute_vlos_doppler(wl_range=model.wl_range).value
print(f"  B_LOS approximation shape: {blos_approx.shape}")
print(f"  V_LOS approximation shape: {vlos_approx.shape}")

blos_approx_patch = blos_approx[y0:y0+H, x0:x0+W]
vlos_approx_patch = vlos_approx[y0:y0+H, x0:x0+W]


Pre-computing physical approximations (WFA B_LOS and Doppler V_LOS) on patch...


  B_LOS approximation shape: (480, 480)
  V_LOS approximation shape: (480, 480)


In [12]:
bz_idx, bz_rrmse_full = model._best_rrmse_index(blos_approx, mhd.od_data['Bz'].value)
vz_idx, vz_rrmse_full = model._best_rrmse_index(vlos_approx, mhd.od_data['Vz'].value)

print(f"Best Bz index: {bz_idx}, RRMSE: {bz_rrmse_full[bz_idx]:.3f}")
print(f"Best Vz index: {vz_idx}, RRMSE: {vz_rrmse_full[vz_idx]:.3f}")

Best Bz index: 8, RRMSE: 1.147
Best Vz index: 20, RRMSE: 6.270


## 4. Training

## 5. Predicting with uncertainty

In [13]:
# Training (dropout active)
model.train()
output = model(stokes_input)  # Normal forward pass with dropout

# Inference WITHOUT uncertainty (fast)
model.eval()
with torch.no_grad():
    predictions = model(stokes_input, return_uncertainty=False)

# Inference WITH uncertainty (30x slower)
model.eval()
with torch.no_grad():
    mean_predictions, std_predictions = model(stokes_input, return_uncertainty=True)
    
print(f"Mean predictions shape: {mean_predictions.shape}")  # (batch, 63)
print(f"Std predictions shape: {std_predictions.shape}")    # (batch, 63)

# Uncertainty per parameter at each optical depth
T_mean = mean_predictions[:, :21]      # (batch, 21)
T_std = std_predictions[:, :21]        # Uncertainty in temperature

Vz_mean = mean_predictions[:, 21:42]
Vz_std = std_predictions[:, 21:42]     # Uncertainty in velocity

Bz_mean = mean_predictions[:, 42:]
Bz_std = std_predictions[:, 42:]   

NameError: name 'stokes_input' is not defined